# Learn action values with a quantum Q-function

Use the same four CartPole features in a value-based agent. Train with replay, inspect the two estimated action values and draw the quantum part of the online network.

Run the cells in order with QASP 0.1.2. See the [tutorial setup guide](../README.md) for installation and optional PDF rendering.

In [ ]:
%matplotlib inline
from pathlib import Path
from importlib.metadata import version
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Image, display

print("QASP", version("qasp"))
np.random.seed(17)
torch.manual_seed(17)

## Configure the experiment

The circuit produces measured features and an affine head produces one Q-value per action. Neural projection weights and variational angles have separate initializers.

In [ ]:
from qasp import Normal, Uniform
from copy import deepcopy
from qasp import create_qfunction, draw_qfunction, DeepQLearning

np.random.seed(2026)
torch.manual_seed(2026)
n_qubits, n_actions = 2, 2
qfunction_config = {
    "feature_map": "NeuralEmbedding",
    "feature_map_config": {"features": 4, "rotations": "Y"},
    "ansatz": "AngleAnsatz",
    "ansatz_config": {
        "n_layers": 1, "rotations": "YZ",
        "include_imprimitive": True,
        "imprimitive": "CZ", "imprimitive_pattern": "chain",
    },
    "observables": ["Z 0", "Z 1"],
    "fc_layer": True, "fc_bias": True,
    "batch_mode": "sequential",
    "init": Normal(0, 0.1),
    "input_scaling_init": Uniform(0.8, 1.2),
    "device": "default.qubit", "diff_method": "backprop",
    "shots": None,
}
trainer_config = {
    "env_name": "CartPole-v1", "seed": 2026,
    "lr": 0.01, "gamma": 0.99,
    "batch_size": 4, "replay_capacity": 64,
    "learning_starts": 4, "train_frequency": 2,
    "gradient_steps": 1, "target_update_interval": 4,
    "epsilon_decay_steps": 32, "max_steps": 8,
    "double_dqn": True,
}

## Construct and inspect the initial model

Run this cell again after editing the configuration to sample fresh parameters.

In [ ]:
qfunction = create_qfunction(
    deepcopy(qfunction_config), n_qubits=n_qubits, n_actions=n_actions
)
probe = torch.tensor([0.12, -0.04, 0.08, 0.02], dtype=torch.float32)
q_values = qfunction(probe)
assert q_values.shape == (n_actions,)
print("Initial Q-values:", q_values.detach())
print("Initial angles:", qfunction.qlayer.params.detach())
print("Initial encoding weights:", qfunction.qlayer.input_scaling_params.detach())

## Train

The agent stores transitions, samples replay minibatches and periodically updates its target network. learning_starts controls when the first optimizer step can occur.

In [ ]:
with DeepQLearning(qfunction, **trainer_config) as agent:
    records = agent.train(num_episodes=4, log_every=0)
    evaluation_returns = agent.evaluate(num_episodes=2, seed=30000)
    update_count = agent.update_count

assert update_count > 0
assert len(records) == 4
assert np.isfinite(evaluation_returns).all()

## Read the results

Compare the action values for a fixed observation with the evaluation returns. A larger Q-value selects the greedy action; the values are on a return scale.

In [ ]:
training_returns = [row["return"] for row in records]
update_losses = [row["loss"] for row in records if row["loss"] is not None]
qfunction.eval()
with torch.no_grad():
    q_values = qfunction(probe)
    greedy_action = int(q_values.argmax().item())

print("Training returns:", training_returns)
print("Update losses:", update_losses)
print("Evaluation returns:", evaluation_returns)
print("Q-values and action:", q_values, greedy_action)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(np.arange(len(training_returns))+1, training_returns, "o-")
ax.set(xlabel="Training episode", ylabel="Episode return")
plt.show()

## Inspect the circuit

Use the trained weights and the same representative input for both renderers. Matplotlib is the default for the module drawing helpers.

In [ ]:
fig, ax = draw_qfunction(qfunction, inputs=probe)
display(fig)
plt.close(fig)

### Preview the paper figure

Request standalone TikZ for a vector PDF. Symbolic labels describe the circuit architecture. Set `symbolic_parameters=False` to inspect the gate angles at this input.

The helper uses temporary build files. To keep the displayed source, explicitly run `Path("circuit.tex").write_text(tex, encoding="utf-8")`, or call `builder.save_tikz(...)` when working with a builder.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import shutil
import subprocess
from IPython.display import Image, display


def show_tikz(source, name="Quantum circuit"):
    """Preview standalone TikZ inline. The optional name is the image description."""
    gs = shutil.which("gs") or shutil.which("gswin64c")
    if not shutil.which("pdflatex") or not gs:
        print("TikZ preview requires pdfLaTeX with Quantikz and Ghostscript.")
        return

    with TemporaryDirectory() as tmp:
        (Path(tmp) / "circuit.tex").write_text(source, encoding="utf-8")
        latex = subprocess.run(
            ["pdflatex", "-no-shell-escape", "-interaction=nonstopmode",
             "-halt-on-error", "circuit.tex"], cwd=tmp, capture_output=True, text=True,
        )
        if latex.returncode:
            raise RuntimeError(latex.stdout[-2000:])
        png = subprocess.check_output(
            [gs, "-q", "-dSAFER", "-sDEVICE=png16m", "-r120", "-o", "-", "circuit.pdf"],
            cwd=tmp,
        )
    display(Image(data=png, alt=name))

In [ ]:
tex = draw_qfunction(
    qfunction, inputs=probe, output="tikz",
    tikz_options={"standalone": True, "symbolic_parameters": True, "row_sep": "0.7cm"},
)
show_tikz(tex, "qfunction")

## Try it

Change target_update_interval while keeping the replay and episode budgets fixed. Compare update count and the loss trajectory.